# 实践 12.1  GBLUP、训练群体关系与跨群体验证

《统计遗传与智能育种学》第十二章 · 电子资源 `12_gs_training_set.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：12_gs_training_set.ipynb。实践生成两个等位基因频率不同的虚拟亚群，并把群体特异标记替代效应的相关设为模拟参数；随后比较A群体内留出（训练210个个体）与全量A预测B（训练300个个体）。中心化和表型均值只用训练集，正则化参数由模拟设定给出。分布距离用训练与测试等位基因频率的均方根（root mean square, RMS）差异描述，在训练样本频率下中心化后，跨训练样本的平均关系按构造为0。这里模拟的是群体特异标记替代效应，实际群体的因果等位基因效应需另行验证。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：固定SEED=1201时，A群体内部留出的预测能力为0.382，GEBV准确度为0.473，真实育种值（TBV）参照下的均方根误差（RMSE）为0.510；训练A→测试B时分别为0.188、0.209和0.703。等位基因频率RMS距离由群体内0.039升到A—B之间0.152。上述差异对应本例的训练规模与群体设定，不能外推为跨群体预测存在固定幅度的性能下降。

## 输入、完整代码、实际输出与解释

输入与可复现条件：两个各300个个体的模拟亚群、300个标记；SEED=1201；目标狭义遗传力h²为0.65，生成标记替代效应的相关参数为0.35；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1201
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1201


In [2]:
from sklearn.model_selection import train_test_split

n, m = 600, 300
n_a = n // 2
p_a = rng.uniform(0.08, 0.45, m)
p_b = np.clip(p_a + rng.normal(0.14, 0.06, m), 0.05, 0.95)
X_a = rng.binomial(2, p_a, size=(n_a, m)).astype(float)
X_b = rng.binomial(2, p_b, size=(n_a, m)).astype(float)

# beta_A/beta_B是群体特异的“标记替代效应”教学真值，不等同于因果等位基因效应。
rho_effect = 0.35
beta_a = rng.normal(0, 1/np.sqrt(m), m)
z = rng.normal(0, 1/np.sqrt(m), m)
beta_b = rho_effect*beta_a + np.sqrt(1-rho_effect**2)*z
tbv_a = (X_a - 2*p_a) @ beta_a
tbv_b = (X_b - 2*p_b) @ beta_b
h2 = 0.65
y_a = tbv_a + rng.normal(0, np.std(tbv_a)*np.sqrt((1-h2)/h2), n_a)
y_b = tbv_b + rng.normal(0, np.std(tbv_b)*np.sqrt((1-h2)/h2), n_a)
print('preset marker-effect correlation', rho_effect,
      'realized', round(float(np.corrcoef(beta_a, beta_b)[0,1]), 3))
print('phenotypic SD A/B', round(float(np.std(y_a)),3), round(float(np.std(y_b)),3))


preset marker-effect correlation 0.35 realized 0.367
phenotypic SD A/B 0.719 0.714


In [3]:
def gblup_fit_predict(Xtr, ytr, Xte, lam=(1-h2)/h2):
    p = Xtr.mean(axis=0)/2
    Wtr = Xtr - 2*p
    Wte = Xte - 2*p
    denom = max(2*np.sum(p*(1-p)), 1e-8)
    G = Wtr @ Wtr.T / denom
    alpha = np.linalg.solve(G + lam*np.eye(len(G)), ytr-ytr.mean())
    Gte = Wte @ Wtr.T / denom
    return ytr.mean() + Gte @ alpha, G

def corr(a, b):
    return float(np.corrcoef(a, b)[0,1])

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))

idx_a = np.arange(n_a)
tr_a, te_a = train_test_split(idx_a, test_size=0.30, random_state=SEED)
pred_same, G = gblup_fit_predict(X_a[tr_a], y_a[tr_a], X_a[te_a])
pred_cross, _ = gblup_fit_predict(X_a, y_a, X_b)

p_train = X_a[tr_a].mean(axis=0)/2
p_holdout = X_a[te_a].mean(axis=0)/2
p_cross = X_b.mean(axis=0)/2
af_rms_same = float(np.sqrt(np.mean((p_train-p_holdout)**2)))
af_rms_cross = float(np.sqrt(np.mean((X_a.mean(axis=0)/2-p_cross)**2)))

res = pd.DataFrame([
    {'scenario':'A random holdout',
     'predictive_ability':corr(pred_same,y_a[te_a]),
     'GEBV_accuracy':corr(pred_same,tbv_a[te_a]),
     'TBV_RMSE':rmse(pred_same,tbv_a[te_a])},
    {'scenario':'train A -> test B',
     'predictive_ability':corr(pred_cross,y_b),
     'GEBV_accuracy':corr(pred_cross,tbv_b),
     'TBV_RMSE':rmse(pred_cross,tbv_b)}
])
print('G shape:', G.shape, 'diag mean:', round(float(np.diag(G).mean()),3))
print('allele-frequency RMS distance: within-A', round(af_rms_same,3),
      'A-vs-B', round(af_rms_cross,3))
print(res.round(3).to_string(index=False))


G shape: (210, 210) diag mean: 1.0
allele-frequency RMS distance: within-A 0.039 A-vs-B 0.152
         scenario  predictive_ability  GEBV_accuracy  TBV_RMSE
 A random holdout               0.382          0.473     0.510
train A -> test B               0.188          0.209     0.703


结果解释：两种场景的训练规模、群体及效应设定不同。本例跨群体预测的GEBV准确度较低，TBV参照下的RMSE较高。这里直接模拟的是群体特异标记替代效应；真实项目中的跨群体下降还可能来自连锁不平衡（LD）相位、等位基因频率、G×E、表型定义和育种阶段差异，不能把本模拟的0.209当成经验常数。

可修改练习：把rho_effect改为0.2、0.6、0.9，另将B群体的频率偏移减半；分别记录GEBV准确度与等位基因频率RMS距离，区分遗传分布距离和效应可迁移性。